# **Méteo, Pollution et Trafic Routier à Paris - Projet Pyhton**

A Paris, la qualité de l'air constitue un enjeu majeur. Par exemple, en moyenne sur l'année 2024, 100% des parisiens ont réspiré un air contenant plus de particules fines que les recommendations de l'OMS. Chaque jour, la circulation routière contribue aux émissions de polluants atmosphériques, tandis que les conditions météorologiques influencent leur dispersion et leur concentration dans l’air. 

Dans ce projet, nous nous interesserons aux différents liens de corrélation et d'interactions entre météorologie, trafic routier et pollution à Paris, et nous nous demanderont : 

Quels sont les liens de corrélation entre la pollution, la météo et trafic routier à Paris ?

In [ ]:
# Liste des imports
import logging
from pathlib import Path

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D
from scipy.interpolate import PchipInterpolator

# Dans le projet
from src.extract import meteo
from src.extract import trafic
from src.extract.trafic import RACINE, lire_config
from src import stockage
from src.clean import trafic as nettoyage_trafic
from src.extract import airparif as extraction_air
from src.clean import airparif as nettoyage_air

# les fonctions de nettoyage journalisent le nombre de lignes concernées : on les affiche
logging.basicConfig(level=logging.INFO, format="%(message)s")

## **I) Téléchargement des datasets**

In [ ]:
RELANCER_EXTRACTION_TRAFIC = False
RELANCER_EXTRACTION_AIR = False
REFAIRE_NETTOYAGE_AIR = True  # False : lit directement les tables nettoyées sur SSP Cloud

Les datasets sur la qualité de l'air à Paris ('airparif') et du trafic routier à Paris ('trafic') sont très volumineux et prennent environ 1h chacun à être téléchargés. Pour gagner du temps, les versions brutes sont enregistrées via SSP Cloud et sont récupérées directement depuis la plateforme.

### **A) Dataset Meteo**

Le premier dataset que nous allons extraire est le plus léger. Il répertorie diverses données sur la météo à Paris. Celui-ci est directement importé via un call API depuis le site internet "https://open-meteo.com/en/docs". Ici, la fonction "meteo_dataset_extraction" est appelée depuis le fichier "src/extract/meteo". L'exploration des données elle se fait directement sur ce fichier.

In [ ]:
import importlib
from src.extract import meteo

importlib.reload(meteo)

df_raw_meteo = meteo.meteo_dataset_extraction()

print("Dimensions :", df_raw_meteo.shape)
display(df_raw_meteo.head())

### Nettoyage & Exploration des données météo à Paris

Tout d'abord, quelques précisions sur la nature des données, et sur la façon dont elles ont été mesurées. Les températures correspondent aux mesures à 2 mètres du sol. Les moyennes
journalières et mensuelles sont calculées à partir des observations horaires.
Les jours et les mois sont définis en **UTC**, comme dans l'extraction.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

df_meteo = df_raw_meteo.copy()

# premier tris des variables qui ne vont pas être très utile au projet

df_meteo = df_meteo.drop(
    columns=[
        "apparent_temperature",
        "dew_point_2m",
        "snow_depth",
        "cloud_cover_low",
        "cloud_cover_mid",
        "cloud_cover_high",
        "soil_temperature_0_to_7cm",
        "soil_moisture_0_to_7cm",
    ]
)

display(df_meteo.isna().sum())
display(df_meteo.dtypes)

# Transformation du format des dates, pour un meilleur gestions de celles-ci avec pandas
df_meteo["time"] = pd.to_datetime(df_meteo["time"], utc=True)
df_meteo = df_meteo.set_index("time", drop=False).sort_index()

temperature_jour = df_meteo["temperature_2m"].resample("D").mean()
temperature_mois = df_meteo["temperature_2m"].resample("MS").mean()

Il n'y a aucune valeur manquante dans le jeu de données. Cela fait donc un problème de moins à régler pour celui-ci. Explorons maintenant un peu ces données ! Nous nous proposons tout d'abord de tracer la courbe de la température moyenne journalière à partir des données téléchargées, puis de réaliser un diagramme en barres représentant la température moyenne de chaque mois. Par la suite, nous tracerons un nuage de points où chaque point représentera, à une heure donnée, la pression atmosphérique en abscisse et la couverture nuageuse en ordonnée, et enfin nous tracerons une courbe représentant la féquence de la pluie en fonction de la pression athmosphérique.

#### Température moyenne par jour

Chaque point de la courbe représente la moyenne des températures horaires d'une journée.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5), constrained_layout=True)
ax.plot(temperature_jour.index, temperature_jour, color="#d97706", linewidth=1.5)

ax.set_title("Température moyenne journalière à Paris")
ax.set_xlabel("Date (UTC)")
ax.set_ylabel("Température moyenne (°C)")
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%Y"))
ax.tick_params(axis="x", rotation=45)
ax.grid(axis="y", alpha=0.25)
ax.margins(x=0.01)
plt.show()

#### Température moyenne par mois

Chaque barre représente la moyenne de toutes les températures horaires du mois.
Les mois sont présentés dans l'ordre chronologique, de septembre 2025 à août 2026.

In [ ]:
mois_fr = [
    "Janv.",
    "Févr.",
    "Mars",
    "Avr.",
    "Mai",
    "Juin",
    "Juil.",
    "Août",
    "Sept.",
    "Oct.",
    "Nov.",
    "Déc.",
]
labels_mois = [
    f"{mois_fr[date.month - 1]}\n{date.year}" for date in temperature_mois.index
]

fig, ax = plt.subplots(figsize=(12, 4.5), constrained_layout=True)
barres = ax.bar(labels_mois, temperature_mois.to_numpy(), color="#2563eb", width=0.65)
ax.bar_label(barres, fmt="%.1f °C", padding=4, fontsize=9)

ax.set_title("Température moyenne mensuelle à Paris")
ax.set_xlabel("Mois (UTC)")
ax.set_ylabel("Température moyenne (°C)")
ax.set_axisbelow(True)
ax.grid(axis="y", alpha=0.25)
ax.margins(y=0.15)
plt.show()

#### Pression en surface et couverture nuageuse

Chaque point correspond aux deux mesures prises à la même heure. La transparence
permet de repérer les zones où les observations se superposent.

In [ ]:
pression_nuages = df_meteo[["surface_pressure", "cloud_cover"]]

fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
ax.scatter(
    pression_nuages["surface_pressure"],
    pression_nuages["cloud_cover"],
    s=12,
    alpha=0.2,
    edgecolors="none",
)

ax.set_title("Pression en surface et couverture nuageuse à Paris")
ax.set_xlabel("Pression en surface (hPa)")
ax.set_ylabel("Couverture nuageuse (%)")
ax.set_ylim(-3, 103)
ax.grid(alpha=0.2)
ax.set_axisbelow(True)
plt.show()

#### Fréquence de la pluie selon la pression atmosphérique

On utilise la pression **au niveau de la mer** (`pressure_msl`), comme sur le
graphique de référence. Une heure est considérée comme pluvieuse si `rain > 0 mm`.
Pour chaque tranche de **5 hPa**, on calcule :

**Fréquence (%) = 100 × nombre d'heures pluvieuses / nombre total d'heures.**

Les points sont placés au centre des tranches. La courbe elle sert de guide
visuel, pour montrer le lien entre pluie et pression. C'est d'ailleurs comme cela que la meteo est prédite par les baromètres mécanique accrochés chez nos grands parents. A noter que les pressions les plus basses ont ici peu d'observations.

In [ ]:
# On créée une nouvelle data frame pandans qui va regrouper les donners dans ces tranches de pressions
df_pluie_pression = df_meteo[["pressure_msl", "rain"]].dropna().copy()
df_pluie_pression["heure_pluvieuse"] = df_pluie_pression["rain"] > 0

# Tranches de 5 hPa couvrant toutes les pressions observées.
pas_pression = 5
pression_min = pas_pression * np.floor(
    df_pluie_pression["pressure_msl"].min() / pas_pression
)
pression_max = pas_pression * np.floor(
    df_pluie_pression["pressure_msl"].max() / pas_pression
)
bornes_pression = np.arange(
    pression_min, pression_max + 2 * pas_pression, pas_pression
)  # le +2 * pas pression est ici pour ne pas manquer les valeurs qu'on aurait pu oublier à cause de l'arrondit au multiple de 5 avec la fonction np.floor
df_pluie_pression["tranche_pression"] = pd.cut(
    df_pluie_pression["pressure_msl"], bins=bornes_pression, right=False
)  # ici, cut va permettre d'attribuer une tranche de pression à chaque heure d'observation de pression

stats_pression_pluie = df_pluie_pression.groupby("tranche_pression").agg(
    nb_heures=(
        "heure_pluvieuse",
        "size",
    ),  # nombre d'heure total observée dans la tranche
    nb_heures_pluvieuses=(
        "heure_pluvieuse",
        "sum",
    ),  # nombre d'heure total dee pluie dans la tranche de pression
    frequence_pluie=(
        "heure_pluvieuse",
        "mean",
    ),  # pourcentage d'heures pluviuse dans la tranche de pression
)  # on regroupe les heures appartennant à une même tranche, en leur créeant des statistiques qui vont nous être utile pour créer le tableau final
stats_pression_pluie["frequence_pluie"] *= 100
stats_pression_pluie["pression_centrale"] = [
    tranche.mid for tranche in stats_pression_pluie.index
]

pressions = stats_pression_pluie["pression_centrale"].to_numpy()
frequences = stats_pression_pluie["frequence_pluie"].to_numpy()

fig, ax = plt.subplots(figsize=(14, 7), constrained_layout=True)
if len(pressions) >= 2:
    pression_lisse = np.linspace(pressions.min(), pressions.max(), 500)
    interpolation = PchipInterpolator(pressions, frequences, extrapolate=False)
    ax.plot(
        pression_lisse,
        interpolation(pression_lisse),
        color="royalblue",
        linewidth=2.5,
        label="Courbe interpolée",
    )  # ici, la méthode permet d'interpoler les points créés, afin de créer une jolie courbe qui va les relier. La courbe n'a en elle même pas une vraie signification scientifique
ax.scatter(
    pressions,
    frequences,
    color="navy",
    s=48,
    zorder=3,
    label="Fréquences observées",
)

ax.set_title(
    "Fréquence de la pluie selon la pression atmosphérique à Paris",
    fontsize=15,
    fontweight="bold",
)
ax.set_xlabel("Pression atmosphérique au niveau de la mer (hPa)", fontsize=12)
ax.set_ylabel("Fréquence des heures pluvieuses (%)", fontsize=12)
ax.set_ylim(0, 100)
ax.set_xlim(bornes_pression[0] - 1, bornes_pression[-1] + 1)
ax.set_yticks(np.arange(0, 101, 20))
ax.grid(alpha=0.2)
ax.set_axisbelow(True)
ax.legend(loc="upper right")
plt.show()

### **B) Dataset trafic**

#### Informations générales

Le dataset est en deux parties: 
- une partie contient pour chaque heure de l'année et pour chaque identifiant d'arc son débit et taux d'occupation
- une autre partie contient pour chaque identifiant d'arc ses caractéristiques géométriques (`geopandas`) 

**Pourquoi regarder le taux d'occupation ?** Le débit `q` (véhicules par heure) est le produit de la densité (véhicules par km) et de la vitesse. En trafic fluide, plus de voitures donne plus de débit. Au-delà d'une densité critique, chaque voiture en plus ralentit les autres : la vitesse chute et le débit plafonne, voire baisse, alors que la densité continue d'augmenter. C'est la saturation qu'on voit ci-dessus. Le taux d'occupation `k` (part du temps où un véhicule est au-dessus du capteur) suit la densité : il continue de monter dans les bouchons, là où le débit ne bouge plus.

Les véh·km mesurent le volume de circulation, c'est l'effet principal sur les émissions. La congestion joue au second ordre, sur ce qu'émet chaque véh·km : en accordéon (freinages, redémarrages, ralenti), un véhicule émet plus de NOx par kilomètre qu'à vitesse stable. Le taux d'occupation révèle ces situations plus chargées.


In [ ]:
CFG = lire_config()

if RELANCER_EXTRACTION_TRAFIC:
    trafic.main()
elif not (RACINE / CFG["chemins"]["raw"] / "trafic").exists():
    stockage.recuperer(CFG, dataset="trafic")

#### Nettoyage

Le nettoyage utilise les fonctions de `src/clean/trafic.py`. Le détail de chaque étape, avec les graphiques qui justifient les choix, est dans `notebooks/analyse_trafic.ipynb`. Les seuils sont dans `config.yaml`. Les différentes étapes de nettoyage qui y sont faite sont les suivantes ;


1. On garde les valeurs `iu_ac` (identifiant de l'arc), `q` (débit, véh/h) et `k` (taux d'occupation, %). `t_1h` marque la **fin** de l'heure : on le décale d'une heure pour obtenir `heure`, le début de période en UTC, et on ne garde que les heures de la période.
2. Le arcs peuvent être représenté dans le dataset plusieurs fois, car d'anciennes versions de données de certains arcs y sont recensé : on garde la plus récente.
3. On retire les arcs sans géométrie, qu'on ne pourrait pas rattacher à une station. Ce qu'on appelle géométrie, c'est la forme de la route stockée comme "valeur" dans le dataset
4. On enlève les séries de zéros de plus de 24 h : le capteur est en panne, ces valeurs passent à `NaN`.
5. On garde les arcs qui ont un débit pour au moins 70 % des heures.
6. On complète le dataset pour qu'il contienne une ligne par arc et par heure, avec `NaN` pour les heures absentes.
7. On ajoute le libellé, la longueur et le centre, ou plutôt le "centroïde" (Lambert-93), de chaque arc.

In [ ]:
SEUIL_ZEROS = CFG["trafic"]["zeros_consecutifs_max"] # 24 par défaut (compromis, voir notebooks/analyse_trafic.ipynb)
SEUIL_DEBIT = CFG["trafic"]["part_sans_debit_max"] # 70% par défaut (idem)
HEURES = nettoyage_trafic.toutes_les_heures(
    CFG["periode"]
)  # heures de la période, début de période en UTC


def resume_trafic(df, etape):
    """Lignes, arcs et valeurs renseignées après une étape du nettoyage."""
    return {
        "étape": etape,
        "lignes": len(df),
        "arcs": df["iu_ac"].nunique(),
        "débits renseignés": df["q"].notna().sum(),
        "taux renseignés": df["k"].notna().sum(),
    }


df_raw_trafic, ref_trafic = nettoyage_trafic.charger_brut(
    RACINE / CFG["chemins"]["raw"] / "trafic"
)
df_trafic = nettoyage_trafic.preparer(df_raw_trafic, HEURES)
etapes_trafic = [resume_trafic(df_trafic, "données brutes")]

# tous les arcs du référentiel, une version par arc (y compris ceux qui seront écartés, pour le tracé géographique plus tard)
arcs_referentiel = nettoyage_trafic.dedoublonner_referentiel(ref_trafic)
df_trafic = nettoyage_trafic.retirer_arcs_sans_geo(df_trafic, arcs_referentiel)
etapes_trafic.append(resume_trafic(df_trafic, "arcs sans géométrie"))

for colonne in ["q", "k"]:
    df_trafic = nettoyage_trafic.retirer_longues_series_de_zeros(
        df_trafic, colonne, SEUIL_ZEROS
    )
etapes_trafic.append(
    resume_trafic(df_trafic, f"séries de zéros de plus de {SEUIL_ZEROS} h")
)

In [ ]:
display(df_trafic)

##### Choix des arcs gardés

Beaucoup d'arcs n'ont presque jamais de débit, d'autres presque toujours. On garde ceux qui ont un débit pour au moins 70 % des heures. Ce choix est un peu arbitraire mais on peut voir la répartition des données par arc et par jour pour se convaincre que la majorité des arcs ont soit très peu de valeurs manquantes, soit énormément. 70% permet donc de couper en majorité des arcs qui avaient bien plus de 30% de valeurs manquantes, et une rue avec 70% des valeurs est encore bien décrite.

In [ ]:
part_sans_debit = nettoyage_trafic.part_heures_sans_debit(
    df_trafic, HEURES
).sort_values()
n_gardes = (part_sans_debit <= SEUIL_DEBIT).sum()

# part des 24 h de chaque jour avec un débit, par arc (arcs triés comme part_sans_debit)
jour = df_trafic["heure"].dt.floor("D")
jours = pd.date_range(jour.min(), jour.max(), freq="D")
dispo = (
    df_trafic["q"]
    .notna()
    .groupby([df_trafic["iu_ac"], jour])
    .sum()
    .unstack(fill_value=0)
)
dispo = dispo.reindex(index=part_sans_debit.index, columns=jours, fill_value=0) / 24
debuts_mois = np.where(jours.day == 1)[0]

fig, (ax_hist, ax_jours) = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
fig.suptitle(
    f"Les arcs ont un débit presque toujours ou presque jamais : {n_gardes} sur {len(part_sans_debit)} sont gardés"
)

ax_hist.hist(part_sans_debit * 100, bins=50, color="#2563eb")
ax_hist.axvline(
    SEUIL_DEBIT * 100,
    color="#dc2626",
    linewidth=1.5,
    label=f"seuil retenu : {SEUIL_DEBIT:.0%}",
)
ax_hist.set_title("Distribution par arc")
ax_hist.set_xlabel("Part des heures sans débit sur la période (%)")
ax_hist.set_ylabel("Nombre d'arcs")
ax_hist.grid(axis="y", alpha=0.25)
ax_hist.legend()

image = ax_jours.imshow(
    dispo.to_numpy(),
    aspect="auto",
    cmap="Blues",
    vmin=0,
    vmax=1,
    interpolation="nearest",
)
ax_jours.axhline(
    n_gardes - 0.5,
    color="#dc2626",
    linewidth=1.5,
    label=f"seuil : {n_gardes} arcs gardés au-dessus",
)
ax_jours.set_title("Détail par arc et par jour")
ax_jours.set_xticks(debuts_mois, jours[debuts_mois].strftime("%m/%Y"), rotation=45)
ax_jours.set_xlabel("Date (UTC)")
ax_jours.set_ylabel("Arcs, triés par part d'heures sans débit")
ax_jours.legend(loc="lower left")
fig.colorbar(
    image, ax=ax_jours, shrink=0.8, label="Part des 24 h du jour avec un débit"
)
plt.show()

df_trafic = nettoyage_trafic.garder_arcs_mesures(df_trafic, SEUIL_DEBIT, HEURES)
etapes_trafic.append(
    resume_trafic(df_trafic, f"arcs sans débit plus de {SEUIL_DEBIT:.0%} du temps")
)

In [ ]:
df_trafic = nettoyage_trafic.completer_grille(df_trafic, HEURES)
etapes_trafic.append(resume_trafic(df_trafic, "grille complète"))

# tables finales du nettoyage, les seules utilisées dans la suite du notebook
df_trafic_clean = nettoyage_trafic.ajouter_infos_arcs(df_trafic, arcs_referentiel)
arcs_clean = arcs_referentiel[arcs_referentiel["iu_ac"].isin(df_trafic_clean["iu_ac"])]
del df_raw_trafic, df_trafic

assert (
    len(df_trafic_clean)
    == df_trafic_clean["iu_ac"].nunique() * df_trafic_clean["heure"].nunique()
)
assert not df_trafic_clean.duplicated(["iu_ac", "heure"]).any()
assert set(arcs_clean["iu_ac"]) == set(df_trafic_clean["iu_ac"])

# même sortie que `uv run python -m src.clean.trafic`
nettoyage_trafic.enregistrer(
    df_trafic_clean, arcs_referentiel, RACINE / CFG["chemins"]["interim"] / "trafic"
)



# À partir de là, uniquement des lignes pour présenter l'évolution du dataset lors du nettoyage
def avec_variation(valeur, precedente):
    """Valeur suivie de sa variation par rapport à l'étape précédente, en %."""
    texte = f"{valeur:,}".replace(",", " ")
    if precedente is None:
        return texte
    if valeur == precedente:
        return f"{texte} (0 %)"
    return f"{texte} ({100 * (valeur - precedente) / precedente:+.2f} %)".replace(
        ".", ","
    )


bilan = pd.DataFrame(etapes_trafic).set_index("étape")
bilan_texte = bilan.astype(object)
for i in range(len(bilan)):
    for colonne in bilan.columns:
        precedente = bilan[colonne].iloc[i - 1] if i > 0 else None
        bilan_texte.iloc[i, bilan.columns.get_loc(colonne)] = avec_variation(
            bilan[colonne].iloc[i], precedente
        )
display(bilan_texte)

part_du_brut = bilan.iloc[-1] / bilan.iloc[0]
print(
    "Part du brut conservée : "
    + " | ".join(f"{colonne} {part:.1%}" for colonne, part in part_du_brut.items())
)
print(
    f"Débit manquant : {df_trafic_clean['q'].isna().mean():.1%} | "
    f"taux d'occupation manquant : {df_trafic_clean['k'].isna().mean():.1%}"
)
display(df_trafic_clean.head())

Dans le tableau, le pourcentage entre parenthèses est la variation par rapport à l'étape précédente. On passe de 23 489 735 lignes et 2 980 arcs à 12 272 496 lignes : 1 531 arcs × 8 016 heures, du 01/11/2025 00:00 au 30/09/2026 23:00 UTC. Le seuil de 70 % retire presque la moitié des arcs mais seulement 11 % des débits renseignés : ce sont surtout des arcs vides. Il manque encore 8,6 % des débits et 23,2 % des taux d'occupation (150 arcs gardés n'ont jamais de `k`) ; rien n'est imputé à ce stade.

Le fichier `data/interim/trafic/trafic.parquet` ne contient pas le tracé des arcs, seulement leur centroïde : le tracé est dans `data/interim/trafic/referentiel_nettoye.parquet`, à joindre par `iu_ac`.

Pour éviter les confusions, la suite du notebook n'utilise que les deux tables finales du nettoyage :
- `df_trafic_clean` : une ligne par arc et par heure (même contenu que `trafic.parquet`) ;
- `arcs_clean` : le tracé des 1 531 arcs gardés, en Lambert-93 (même contenu que `referentiel_nettoye.parquet`).

`arcs_referentiel` contient tous les arcs du référentiel : il ne sert qu'au fond des cartes.

### Exploration des données de trafic nettoyées

#### Distribution du débit

In [ ]:
display(df_trafic_clean["q"].describe().to_frame("débit (véh/h)").T.round(0))

fig, ax = plt.subplots(figsize=(12, 4.5), constrained_layout=True)
ax.hist(df_trafic_clean["q"].dropna(), bins=80, color="#2563eb")
ax.set_yscale("log")
ax.axvline(
    df_trafic_clean["q"].median(),
    color="#dc2626",
    linewidth=1.5,
    label=f"médiane : {df_trafic_clean['q'].median():.0f} véh/h",
)
ax.set_title(
    "La plupart des débits sont sous 2 000 véh/h, avec un second groupe vers 5 000 véh/h (périphérique)"
)
ax.set_xlabel("Débit (véh/h)")
ax.set_ylabel("Nombre d'heures × arcs (échelle log)")
ax.grid(axis="y", alpha=0.25)
ax.legend()
plt.show()

#### Rythme de la semaine

Profils moyens pour chaque heure de la semaine, sur toute la période, en **heure de Paris** (en UTC, les pointes se décalent d'une heure entre l'hiver et l'été et se brouillent), sur l'ensemble des arcs et sur le périphérique seul.

- **Débit** : moyenne pondérée par la longueur des arcs mesurés à cette heure, c'est-à-dire les véh·km divisés par les km mesurés, car ce sont les véh·km qui comptent pour les émissions.
- **Taux d'occupation** : à chaque heure, on fait la moyenne des seuls arcs mesurés (23 % des valeurs manquent et 150 arcs n'en ont jamais)

In [ ]:
jours_semaine = ["Lun.", "Mar.", "Mer.", "Jeu.", "Ven.", "Sam.", "Dim."]
heure_locale = df_trafic_clean["heure"].dt.tz_convert("Europe/Paris")
heure_semaine = heure_locale.dt.dayofweek * 24 + heure_locale.dt.hour  # 0 = lundi 0 h, heure de Paris jusqu'à 168
peripherique = df_trafic_clean["libelle"].str.match(r"P[EI]_")  # périphérique extérieur et intérieur
q, k, longueur = df_trafic_clean["q"], df_trafic_clean["k"], df_trafic_clean["longueur_m"]

# débit pondéré : véhicules × mètres divisés par la longueur des seuls arcs mesurés à cette heure
vehicules_metres = q * longueur
metres_mesures = longueur.where(q.notna())
profil_semaine = pd.DataFrame({
    "q": vehicules_metres.groupby(heure_semaine).sum() / metres_mesures.groupby(heure_semaine).sum(),
    "q_peripherique": (vehicules_metres[peripherique].groupby(heure_semaine[peripherique]).sum()
                       / metres_mesures[peripherique].groupby(heure_semaine[peripherique]).sum()),
    "k": k.groupby(heure_semaine).mean(),
    "k_peripherique": k[peripherique].groupby(heure_semaine[peripherique]).mean(),
})

fig, (ax_q, ax_k) = plt.subplots(2, 1, figsize=(12, 8), sharex=True, constrained_layout=True)
nom_peripherique = f"périphérique ({df_trafic_clean.loc[peripherique, 'iu_ac'].nunique()} arcs)"

ax_q.plot(profil_semaine.index, profil_semaine["q_peripherique"], color="#d97706", linewidth=1.8, label=nom_peripherique)
ax_q.plot(profil_semaine.index, profil_semaine["q"], color="#2563eb", linewidth=1.8, label="tous les arcs")
ax_q.set_title("Le débit plafonne de 7 h à 19 h, sur le périphérique comme sur l'ensemble des arcs")
ax_q.set_ylabel("Débit moyen pondéré par la longueur (véh/h)")

ax_k.plot(profil_semaine.index, profil_semaine["k_peripherique"], color="#d97706", linewidth=1.8, label=nom_peripherique)
ax_k.plot(profil_semaine.index, profil_semaine["k"], color="#2563eb", linewidth=1.8, label="tous les arcs")
ax_k.set_title("Le taux d'occupation fait ressortir les pointes de 8 h et 17 h en semaine, surtout sur le périphérique")
ax_k.set_ylabel("Taux d'occupation moyen (%)")
ax_k.set_xlabel("Heure de la semaine (heure de Paris)")

for ax in (ax_q, ax_k):
    ax.set_ylim(bottom=0)
    for jour in range(1, 7):
        ax.axvline(24 * jour, color="grey", linewidth=0.8, alpha=0.5)
    for heure in range(0,168):
        ax.axvline(heure, color="grey", linewidth=0.4, alpha=0.2)    #.grid() ne marche pas avec les xticks des jours
    ax.set_xticks(range(0, 168, 6), minor=True)
    ax.grid(axis="y", alpha=0.25)
    ax.grid(axis="x", which="minor", alpha=0.15)
ax_k.set_xticks(range(12, 168, 24), jours_semaine)
ax_k.set_xlim(0, 167)
fig.legend(*ax_q.get_legend_handles_labels(), loc="outside lower center", ncols=2)
plt.show()

#### Carte du débit moyen

Chaque arc gardé est coloré selon son débit moyen sur la période ; les arcs en gris n'ont pas assez de données de débit (un débit pour moins de 70 % des heures, ou aucun) et ont été écartés au nettoyage.

In [ ]:
q_moyen = df_trafic_clean.groupby("iu_ac")["q"].mean().rename("q_moyen")
carte = arcs_clean.merge(q_moyen, left_on="iu_ac", right_index=True).sort_values(
    "q_moyen"
)  # forts débits dessinés en dernier
arcs_ecartes = arcs_referentiel[~arcs_referentiel["iu_ac"].isin(arcs_clean["iu_ac"])]

fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)
arcs_ecartes.plot(ax=ax, color="#d1d5db", linewidth=0.6)
carte.plot(
    ax=ax,
    column="q_moyen",
    cmap="viridis",
    norm=LogNorm(),
    linewidth=1.5,
    legend=True,
    legend_kwds={"label": "Débit moyen (véh/h, échelle log)", "shrink": 0.6},
)
ax.legend(
    handles=[
        Line2D(
            [0],
            [0],
            color="#d1d5db",
            linewidth=2,
            label=f"arcs écartés : pas assez de données de débit ({len(arcs_ecartes)})",
        )
    ],
    loc="lower left",
    frameon=False,
)
ax.set_title("Le périphérique et les grands axes concentrent le trafic")
ax.set_axis_off()
plt.show()

#### Valeurs manquantes

Part des débits manquants chaque jour, sur les arcs gardés.

In [ ]:
manquant_jour = (
    df_trafic_clean["q"].isna().groupby(df_trafic_clean["heure"].dt.floor("D")).mean()
    * 100
)

fig, ax = plt.subplots(figsize=(12, 4.5), constrained_layout=True)
ax.plot(manquant_jour.index, manquant_jour.to_numpy(), color="#2563eb", linewidth=1.5)
ax.axhline(
    manquant_jour.median(),
    color="grey",
    linestyle="--",
    linewidth=1,
    label=f"médiane : {manquant_jour.median():.1f} %",
)
ax.set_ylim(0, 100)
ax.set_title(
    "Peu de débits manquent au quotidien, sauf pendant cinq coupures de la collecte"
)
ax.set_xlabel("Date (UTC)")
ax.set_ylabel("Débits manquants (%)")
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%Y"))
ax.tick_params(axis="x", rotation=45)
ax.grid(axis="y", alpha=0.25)
ax.legend()
plt.show()

print(
    "Jours avec plus de 50 % de débits manquants :",
    ", ".join(manquant_jour[manquant_jour > 50].index.strftime("%d/%m/%Y")),
)

#### Conclusion

**Cycle hebdomadaire:** Les données de débit permettent de voir l'alternance jour/nuit mais pas, au sein d'une journée, les heures de pointe attendues (migrations pendulaires à ≈8h et ≈18h). C'est parce que les axes sont saturés en moyenne toute la journée de 6h à 17h(impossible de monter à plus de ≈1200 vh/h). On peut tout de même voir ces heures de pointes avec le taux d'occupation qui, lorsque le débit est maximal, est une mesure du taux de congestion.

**Répartition du débit:** Le débit varie beaucoup d'un arc à l'autre (médiane 297 véh/h, jusqu'à près de 12 000 sur le périphérique). Attention à bien pondérer par la longueur des arcs car ce qui compte pour les émissions sont bien les véhicules-kilomètres. Les arcs les plus longs sont ceux avec le plus de débit (périphérique).

**Données manquantes:** Il manque 6,3% des données de débit sur le jour médian, mais cinq épisodes n'ont presque aucune mesures. Il faudra en tenir compte.

### **C) Dataset qualité de l'air**

Passons enfin au dernier dataset. Il nous donne les mesures de pollutions des stations Airparif. Les gazs poluants étudiés sont le NO2 (dioxyde d'azote), PM2.5 (particules fines d'un diamètre plus petit ou égal à 2 µm), PM10 (particules fines d'un diamètre plus petit ou égal à 10 µm), O3 (Ozone), NO (monoxyde d'azote) et NOx (groupe de gaz polluants composés d'azote et d'oxygène), en µg/m³. Le flux d'Airparif ne garde que les dernières 24 h ; l'historique vient des [fichiers horaires du LCSQA sur data.gouv.fr](https://www.data.gouv.fr/datasets/5b98b648634f415309d52a50), qui reprennent les mesures Airparif **en UTC** avec leurs indicateurs de validité, plus les coordonnées des stations. Aucune clé d'API n'est nécessaire.

Les fichiers bruts (un par jour) et les tables nettoyées sont figés sur SSP Cloud, dans le dossier public `diffusion` de `mathismarsoin`. Par défaut, le notebook récupère les fichiers bruts et refait le nettoyage ; avec `REFAIRE_NETTOYAGE_AIR = False`, il lit directement les tables nettoyées (moins d'une seconde, sans identifiants). Pour repartir de la source, passer `RELANCER_EXTRACTION_AIR` à `True` (environ 45 min la première fois, avec un cache par jour). Le code est dans `src/extract/airparif.py` et `src/clean/airparif.py` ; les contrôles détaillés de l'extraction sont dans `notebooks/qualite_air.ipynb`.

In [ ]:
if RELANCER_EXTRACTION_AIR:
    extraction_air.extraire(CFG)
elif (
    REFAIRE_NETTOYAGE_AIR and not (RACINE / CFG["chemins"]["raw"] / "airparif").exists()
):
    stockage.recuperer(CFG, dataset="airparif")

#### Nettoyage

On effectue d'abord un rapide nettoyage du dataset. Le nettoyage utilise les fonctions de `src/clean/airparif.py`. les seuils sont dans `config.yaml`. Aucun trou n'est bouché : une valeur fausse devient `NaN`. On effectue les modifications suivantes :

1. On ne garde que les jours de la période et les six polluants étudiés (le CO, en mg/m³, est retiré).
2. On enlève les doublons, en ne conservant que les données qui ont le meilleur code qualité.
3. Certaines valeurs dont invalidée par Airparif (validité `-1`). D'autres valeurs peuvent être négative ou nulle (sauf O₃ et NO, qui peuvent valoir 0). On les remplace par des `NaN`.
4. Les mesures pas encore validées par Airparif (code `R`) sont gardées.
5. Les capteur figé qui affichent la même valeur ≥ 5 µg/m³ pendant au moins 6 h sont considérés comme défaillants. Les valeurs associées sont donc remplacés par des `NaN`. Les pic isolé (plus de 5 fois la médiane sur 24 h) : gardé mais signalé.
6. On réorganise ensuite les données : une ligne par station et par heure, une colonne par polluant, sur une grille horaire complète.

In [ ]:
if REFAIRE_NETTOYAGE_AIR or RELANCER_EXTRACTION_AIR:
    tables_air = nettoyage_air.nettoyer(CFG)
    nettoyage_air.sauvegarder(
        tables_air, CFG
    )  # même sortie que `uv run python -m src.clean.airparif`
else:
    # tables déjà nettoyées, figées dans le dossier public diffusion : lecture directe, sans identifiants
    tables_air = nettoyage_air.charger_depuis_ssp_cloud(CFG)

# tables finales du nettoyage, les seules utilisées dans la suite du notebook
df_air = tables_air["mesures"]
df_stations_air = tables_air["stations"]

assert len(df_air) == df_air["station"].nunique() * df_air["heure"].nunique()
assert not df_air.duplicated(["station", "heure"]).any()
print(
    f"df_air : {len(df_air):,} lignes = {df_air['station'].nunique()} stations × {df_air['heure'].nunique()} heures, "
    f"du {df_air['heure'].min():%d/%m/%Y %H:%M} au {df_air['heure'].max():%d/%m/%Y %H:%M} UTC".replace(
        ",", " "
    )
)

with pd.option_context("display.max_colwidth", None):
    display(tables_air["decisions"])

Sur 1 368 296 lignes brutes, le nettoyage met à `NaN` 27 922 valeurs invalidées par Airparif, 9 553 valeurs négatives (petits écarts de l'appareil autour de zéro, surtout pour le NO), 32 zéros et 19 heures de capteur figé. Les valeurs négatives ne touchent que les heures très peu polluées : ces trous ne sont pas aléatoires (MNAR), à garder en tête si on les impute. 3 140 pics isolés sont signalés dans `tables_air["anomalies"]` mais gardés : un pic de pollution peut être réel.

La suite du notebook n'utilise que deux tables :
- `df_air` : une ligne par station et par heure (UTC, début de période), une colonne par polluant (même contenu que `data/interim/airparif/mesures.parquet`) ;
- `df_stations_air` : une ligne par station, avec son type (`trafic` : bord de route, `fond` : loin des axes) et ses coordonnées.

In [ ]:
display(df_air)

### Exploration des données de qualité de l'air

#### Où sont les stations ?

Les coordonnées des stations (WGS 84) sont projetées en Lambert-93, comme les arcs de trafic, pour mesurer la distance de chaque station à l'arc gardé le plus proche.

In [ ]:
import geopandas as gpd

points_air = gpd.GeoDataFrame(
    df_stations_air,
    geometry=gpd.points_from_xy(
        df_stations_air["longitude"], df_stations_air["latitude"]
    ),
    crs="EPSG:4326",
).to_crs(arcs_clean.crs)
points_air = points_air.sjoin_nearest(
    arcs_clean[["iu_ac", "geometry"]], distance_col="distance_arc_m"
)
points_air = points_air.drop_duplicates(
    "station"
)  # deux arcs à égale distance : un seul suffit
proches = points_air["distance_arc_m"] <= 500
print(
    f"{proches.sum()} stations sur {len(points_air)} sont à moins de 500 m d'un arc gardé :"
)
display(
    points_air.loc[proches, ["station", "nom", "influence", "distance_arc_m"]]
    .sort_values("distance_arc_m")
    .round({"distance_arc_m": 0})
)

fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)
arcs_clean.plot(ax=ax, color="#d1d5db", linewidth=0.8)
couleurs_influence = {"trafic": "#dc2626", "fond": "#2563eb"}
for influence, groupe in points_air[proches].groupby("influence"):
    groupe.plot(
        ax=ax,
        color=couleurs_influence[influence],
        markersize=60,
        edgecolor="white",
        label=f"station {influence} ({len(groupe)})",
    )
ax.legend(loc="lower left", frameon=False)
ax.set_title("Les stations Airparif à moins de 500 m d'un arc de trafic gardé")
ax.set_axis_off()
plt.show()

In [ ]:
stations_proches = points_air.loc[proches, "station"]
df_air_paris = df_air[df_air["station"].isin(stations_proches)].copy()

Les stations Airparif couvrent toute l'Île-de-France, alors que les comptages de trafic ne couvrent que Paris : seules les stations proches d'un arc gardé pourront être reliées au trafic. Les autres restent utiles comme points de comparaison (fond régional). On créer alors cette data frame "df_air_paris" pour ne conserver que les données relatives à Paris.

#### Rythme de la semaine du NO₂

NO₂ moyen pour chaque heure de la semaine (UTC), séparément pour les stations trafic et les stations de fond, sur toute la période.

In [ ]:
df_air_influence = df_air.merge(df_stations_air[["station", "influence"]], on="station")
heure_semaine_air = (
    df_air_influence["heure"].dt.dayofweek * 24 + df_air_influence["heure"].dt.hour
)  # 0 = lundi 0 h
profil_no2 = (
    df_air_influence.groupby(["influence", heure_semaine_air])["no2"]
    .mean()
    .unstack("influence")
)
display(
    df_air_influence.groupby("influence")[["no2", "pm25", "pm10", "o3"]]
    .mean()
    .round(1)
    .rename(columns={"no2": "NO₂", "pm25": "PM₂.₅", "pm10": "PM₁₀", "o3": "O₃"})
)

fig, ax = plt.subplots(figsize=(12, 4.5), constrained_layout=True)
for influence in ["trafic", "fond"]:
    ax.plot(
        profil_no2.index,
        profil_no2[influence].to_numpy(),
        color=couleurs_influence[influence],
        linewidth=1.8,
        label=f"stations {influence}",
    )
for jour_air in range(1, 7):
    ax.axvline(24 * jour_air, color="grey", linewidth=0.8, alpha=0.5)
ax.set_xticks(range(12, 168, 24), jours_semaine)
ax.set_xlim(0, 167)
ax.set_title(
    "Le NO₂ suit le même rythme que le trafic, avec un niveau presque deux fois plus haut au bord des routes"
)
ax.set_xlabel("Heure de la semaine (UTC)")
ax.set_ylabel("NO₂ moyen (µg/m³)")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.25)
plt.show()

En moyenne sur la période, le NO₂ vaut 32 µg/m³ aux stations trafic contre 17 µg/m³ aux stations de fond, et l'écart se creuse aux heures de pointe (jusqu'à 44 µg/m³ en moyenne au bord des routes). Le week-end, le NO₂ baisse d'environ 14 % dans les deux cas. L'ozone, lui, n'est mesuré qu'en fond : il ne suit pas le trafic (il se forme au soleil, loin des sources).

#### Valeurs manquantes

Part des heures de la période où chaque station a une valeur, polluant par polluant. Une case vide signifie que la station ne mesure pas ce polluant (ce n'est pas un trou).

In [ ]:
polluants_air = list(CFG["airparif"]["polluants"].values())
couverture_air = tables_air["couverture"]
part_valide_air = couverture_air.pivot(
    index="station", columns="polluant", values="part_valide"
).reindex(columns=polluants_air)
part_valide_air.index = (
    part_valide_air.index
    + " "
    + df_stations_air.set_index("station").loc[part_valide_air.index, "nom"]
)
part_valide_air.style.background_gradient(cmap="RdYlGn", vmin=0, vmax=1).format(
    "{:.0%}", na_rep=""
)

#### Ce qu'on retient pour la suite

- `df_air` contient 46 stations × 8 016 heures (01/11/2025 → 30/09/2026, UTC), la même grille horaire que `df_trafic_clean` : les deux tables se joignent directement sur `heure`.
- Pour la fusion spatiale, on relie chaque station aux arcs de trafic proches (par exemple dans un rayon de quelques centaines de mètres), à partir de `points_air`, déjà en Lambert-93.
- Le NO₂ est le polluant le plus mesuré (36 stations sur 46, dont 10 stations trafic) et le plus lié au trafic : c'est lui qui servira à répondre à la question. Les PM₂.₅, les PM₁₀ et l'O₃ ne sont mesurés que par une partie des stations.
- Les PM₁₀ des Halles (FR04055) s'arrêtent le 31/12/2025 (16 % de la période) : série signalée, pas supprimée.

## **II) Croisement des données et Analyse**

### **A) Retard de la détection de la pollution sur l'émission**

L'idée est que la pollution n'est pas immédiatement détectée à l'émission : il faut d'abord qu'elle se diffuse dans l'air jusqu'aux stations de mesure. Celles proches du trafic devraient avoir une détection quasi immédiate (en tout cas en moins d'une heure) ; dans cette section, on regarde si le retard est plus important pour les autres. Cela poserait problème car lors de l'étude des corrélations, nous trouverions des valeurs plus faibles que les vraies valeurs en comparant naïvement heure par heure; si le retard observé est supérieur à 1h, il faudra appliquer un décalage entre les séries.

#### Étape 1 : profils agrégés du trafic et du NO₂

On compare le profil hebdomadaire du trafic à celui du NO₂, en agrégeant en deux temps :
- **à chaque heure, sur tout Paris.** Pour le trafic, on calcule les **véhicules-kilomètres** (véh·km), car les émissions sont proportionnelles à la distance parcourue : débit × longueur de chaque arc. Pour qu'une panne de capteurs ne fasse pas baisser le total, on prend la moyenne des débits pondérée par la longueur des arcs mesurés, multipliée par la longueur totale du réseau (306 km). Pour le NO₂, on fait la moyenne des stations situées dans la zone des comptages, séparément pour les stations trafic et de fond ;
- **pour chaque heure de la semaine, la médiane sur les semaines**, pour ne pas être tiré par les épisodes de pollution ou les jours de coupure.

Les heures sont en **heure de Paris** : en UTC, les pointes se décalent d'une heure entre l'hiver et l'été. Le trafic est sur l'axe de gauche et le NO₂ sur celui de droite, **les deux partant de 0** : on lit les vraies valeurs et aucune courbe n'est déformée par une normalisation.

In [ ]:
# stations qui mesurent le NO2 dans la zone couverte par les arcs de comptage (Paris)
zone_comptages = arcs_clean.union_all().convex_hull #enveloppe convexe des arcs 
points_stations = gpd.GeoSeries(gpd.points_from_xy(df_stations_air["longitude"], df_stations_air["latitude"]),
                                index=df_stations_air.index, crs=4326).to_crs(arcs_clean.crs)
stations_paris = df_stations_air[points_stations.within(zone_comptages)
                                 & df_stations_air["polluants_mesures"].str.contains("no2")]     # On garde les mesures de NOx (émis par les voitures) dans Paris
for influence, noms in stations_paris.groupby("influence")["nom"]:
    print(f"{influence} ({len(noms)}) : {', '.join(noms)}")

# véh·km par heure : débit moyen pondéré par la longueur des arcs mesurés, ramené à tout le réseau
longueur_km = df_trafic_clean["longueur_m"] / 1000
longueur_reseau_km = df_trafic_clean.drop_duplicates("iu_ac")["longueur_m"].sum() / 1000
par_heure = (df_trafic_clean
             .assign(vkm=df_trafic_clean["q"] * longueur_km,
                     km_mesures=longueur_km.where(df_trafic_clean["q"].notna()))
             .groupby("heure")[["vkm", "km_mesures"]].sum())
vkm_heure = par_heure["vkm"] / par_heure["km_mesures"] * longueur_reseau_km

peripherique = df_trafic_clean["libelle"].str.match(r"P[EI]_")  # périphérique extérieur et intérieur
no2_heure = (df_air.merge(stations_paris[["station", "influence"]], on="station")
             .groupby(["heure", "influence"])["no2"].mean().unstack())

# une ligne par heure de la période (UTC), toutes les séries de Paris côte à côte
series_heure = pd.DataFrame({
    "vkm": vkm_heure,
    "k": df_trafic_clean.groupby("heure")["k"].mean(),
    "q_peripherique": df_trafic_clean[peripherique].groupby("heure")["q"].mean(),
    "k_peripherique": df_trafic_clean[peripherique].groupby("heure")["k"].mean(),
    "no2_trafic": no2_heure["trafic"],
    "no2_fond": no2_heure["fond"],
    "vent": df_meteo["wind_speed_10m"],
}).reindex(HEURES)
print(f"Réseau : {longueur_reseau_km:.0f} km, dont {df_trafic_clean.loc[peripherique, 'iu_ac'].nunique()} arcs du périphérique")

# profils en heure de Paris : médiane sur les semaines
heure_paris = series_heure.index.tz_convert("Europe/Paris")
ouvre = heure_paris.dayofweek < 5
dimanche = heure_paris.dayofweek == 6
profil_semaine = series_heure.groupby(heure_paris.dayofweek * 24 + heure_paris.hour).median()
profil_ouvre = series_heure[ouvre].groupby(heure_paris[ouvre].hour).median()
profil_dimanche = series_heure[dimanche].groupby(heure_paris[dimanche].hour).median()

In [ ]:
COULEUR_TRAFIC, COULEUR_NO2_TRAFIC, COULEUR_NO2_FOND = "#374151", "#dc2626", "#2563eb"


def tracer_vkm_no2(ax, profil):
    """Véh·km à gauche, NO2 des deux types de stations à droite, les deux axes partant de 0."""
    ax.plot(profil.index, profil["vkm"] / 1000, color=COULEUR_TRAFIC, linewidth=2.2, label="trafic (véh·km)")
    ax.set_ylim(0, profil["vkm"].max() / 1000 * 1.1)
    ax.set_ylabel("Trafic (milliers de véh·km par heure)")
    ax.grid(axis="y", alpha=0.25)
    ax_no2 = ax.twinx()
    ax_no2.plot(profil.index, profil["no2_trafic"], color=COULEUR_NO2_TRAFIC, linewidth=1.8, label="NO₂ des stations trafic")
    ax_no2.plot(profil.index, profil["no2_fond"], color=COULEUR_NO2_FOND, linewidth=1.8, label="NO₂ des stations de fond")
    ax_no2.set_ylim(0, profil[["no2_trafic", "no2_fond"]].max().max() * 1.1)
    ax_no2.set_ylabel("NO₂ (µg/m³)")
    return ax.get_lines() + ax_no2.get_lines()


fig, (ax_sem, ax_jour) = plt.subplots(2, 1, figsize=(12, 8.5), constrained_layout=True)
fig.suptitle("Le trafic forme un plateau de 8 h à 19 h, alors que le NO₂ a deux pics, le matin et le soir")

lignes = tracer_vkm_no2(ax_sem, profil_semaine)
for jour in range(1, 7):
    ax_sem.axvline(24 * jour, color="grey", linewidth=0.8, alpha=0.5)
ax_sem.set_xticks(range(12, 168, 24), jours_semaine)
ax_sem.set_xlim(0, 167)
ax_sem.set_title("Semaine type")
ax_sem.set_xlabel("Heure de la semaine (heure de Paris)")

tracer_vkm_no2(ax_jour, profil_ouvre)
ax_jour.set_xticks(range(0, 24, 2))
ax_jour.set_xlim(0, 23)
ax_jour.set_title("Jour ouvré type (lundi au vendredi)")
ax_jour.set_xlabel("Heure de la journée (heure de Paris)")

fig.legend(handles=lignes, loc="outside lower center", ncols=3)
plt.show()

Le trafic **sature** : les véh·km montent entre 5 h et 8 h, puis restent au même niveau jusqu'à 19 h, sans pointe du matin ni du soir. Le NO₂, lui, a bien deux pics (vers 8 h et en soirée) et un creux l'après-midi. Le débit ne voit donc pas tout ce qui se passe sur la route.

**Pourquoi regarder le taux d'occupation ?** Le débit `q` (véhicules par heure) est le produit de la densité (véhicules par km) et de la vitesse. En trafic fluide, plus de voitures donne plus de débit. Au-delà d'une densité critique, chaque voiture en plus ralentit les autres : la vitesse chute et le débit plafonne, voire baisse, alors que la densité continue d'augmenter. C'est la saturation qu'on voit ci-dessus. Le taux d'occupation `k` (part du temps où un véhicule est au-dessus du capteur) suit la densité : il continue de monter dans les bouchons, là où le débit ne bouge plus.

Les véh\*km mesurent le volume de circulation, c'est l'effet principal sur les émissions. La congestion joue au second ordre, sur ce qu'émet chaque véh\*km : en accordéon (freinages, redémarrages, ralenti), un véhicule émet plus de NOx par kilomètre qu'à vitesse stable. Le taux d'occupation révèle ces situations plus chargées.

In [ ]:


display(profil_ouvre.loc[[8, 11, 13, 17], ["vkm", "k", "q_peripherique", "k_peripherique"]]
        .round(1).rename_axis("heure (Paris)"))

Sur le périphérique, le débit reste entre 4 600 et 4 850 véh/h de 7 h à 19 h, alors que le taux d'occupation passe de 26 % à 8 h à 20 % en fin de matinée, puis remonte à 27 % à 17 h : ce sont les heures de pointe. Sur l'ensemble des arcs, l'effet existe mais il est plus faible (12 % à 8 h, 11 % à 13 h, 13 % à 17 h).
**Voir figure "Rythme de la semaine" dans l'exploration du dataset trafic (I.B) pour plus de détails.**

La congestion n'explique pas tout : entre 8 h et 14 h, le NO₂ de fond baisse de 60 %, le taux d'occupation du périphérique de 20 % seulement. En journée, le sol chauffe et l'air se mélange sur une plus grande hauteur, ce qui dilue la pollution ; le soir, cette couche de mélange s'abaisse et le NO₂ s'accumule. Comparer les pics des deux courbes mélange donc l'effet du trafic, l'effet de la météo et un éventuel retard. L'étape suivante sert à les séparer.

#### Étape 2 : jour ouvré moins dimanche

Le dimanche a le même soleil et le même cycle de mélange de l'air qu'un jour ouvré, mais la congestion arrive beaucoup plus tard dans la journée (après 14h). Pour chaque heure de la matinée, la différence « jour ouvré type − dimanche type » retire donc le cycle de la météo et ne garde que ce qui vient du surplus de trafic. S'il y a un retard entre l'émission et la mesure, le pic du surplus de NO₂ doit arriver après celui du surplus de trafic. 

In [ ]:
ecart_ouvre_dimanche = profil_ouvre - profil_dimanche

fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)
ax.plot(ecart_ouvre_dimanche.index, ecart_ouvre_dimanche["vkm"] / 1000, color=COULEUR_TRAFIC, linewidth=2.2,
        marker="o", markersize=3, label="trafic (véh·km)")
limite = ecart_ouvre_dimanche["vkm"].abs().max() / 1000 * 1.1
ax.set_ylim(-limite, limite)
ax.set_ylabel("Écart de trafic (milliers de véh·km par heure)")
ax_no2 = ax.twinx()
for colonne, couleur, nom in [("no2_trafic", COULEUR_NO2_TRAFIC, "NO₂ des stations trafic"),
                              ("no2_fond", COULEUR_NO2_FOND, "NO₂ des stations de fond")]:
    ax_no2.plot(ecart_ouvre_dimanche.index, ecart_ouvre_dimanche[colonne], color=couleur, linewidth=1.8,
                marker="o", markersize=3, label=nom)
limite = ecart_ouvre_dimanche[["no2_trafic", "no2_fond"]].abs().max().max() * 1.1
ax_no2.set_ylim(-limite, limite)  # axes symétriques : le zéro est à la même hauteur des deux côtés
ax_no2.set_ylabel("Écart de NO₂ (µg/m³)")
ax.set_xticks(range(0, 24))
ax.set_xlim(0, 23)
ax.set_xlabel("Heure de la journée (heure de Paris)")
ax.grid(axis="y", alpha=0.25)
ax.set_title("Le surplus de trafic et le surplus de NO₂ des jours ouvrés culminent ensemble, entre 7 h et 8 h")
ax.legend(handles=ax.get_lines() + ax_no2.get_lines(), loc="upper right")
ax.axhline(0, color="grey", linewidth=0.8)
plt.show()

print("Heure du maximum de l'écart :",
      ", ".join(f"{c} {h} h" for c, h in ecart_ouvre_dimanche[["vkm", "no2_trafic", "no2_fond"]].idxmax().items()))

# corrélation entre l'écart de trafic décalé de d heures et l'écart de NO2
decalages = range(-3, 4)
correlation_decalage = pd.DataFrame(
    {colonne: [ecart_ouvre_dimanche["vkm"].shift(d).corr(ecart_ouvre_dimanche[colonne]) for d in decalages]
     for colonne in ["no2_trafic", "no2_fond"]},
    index=pd.Index(decalages, name="retard du NO₂ sur le trafic (h)"),
)
display(correlation_decalage.round(3))

Le surplus de trafic des jours ouvrés culmine à 7 h et celui du NO₂ à 8 h, mais les deux heures sont presque à égalité pour le trafic (220 000 et 215 000 véh·km de plus que le dimanche). La corrélation entre les deux écarts est maximale et presque identique pour un retard de 0 h (0,96) et de 1 h (0,95), et retombe de la même façon de part et d'autre (environ 0,8 à −1 h et à +2 h) : le retard se situe entre 0 et 1 h. **Le NO₂ suit donc le trafic avec un retard inférieur à une heure, aux stations trafic comme aux stations de fond** : à la résolution horaire des données, on ne peut pas le mesurer plus finement.

C'est cohérent avec la physique : près d'un axe, la pollution atteint le capteur en quelques secondes ou minutes ; avec un vent de 2 à 4 m/s, l'air traverse Paris en environ une heure, et les stations de fond reçoivent la pollution de toute la ville, y compris des arcs tout proches. Le soir, l'écart est presque nul pour le trafic comme pour le NO₂ : **la pointe de NO₂ du soir vient de la météo** (couche de mélange qui s'abaisse), pas du retour du travail.

### **B) Météo et Pollution**


Nous proposons, dans un premier temps, d’étudier le lien entre la météo et la pollution atmosphérique. On pourrait s’attendre à plusieurs effets. Une journée venteuse pourrait disperser la pollution parisienne vers l’extérieur de la capitale, ce qui entraînerait une baisse des concentrations de certaines particules mesurées dans l’air. De même, les jours pluvieux pourraient inciter certaines personnes à se déplacer en voiture plutôt qu’à pied. À l’inverse, une journée ensoleillée pourrait encourager les Parisiens à privilégier la marche. La température pourrait également jouer un rôle dans le choix des modes de déplacement et, indirectement, influencer les niveaux de pollution.

A noter que nous devons traiter les valeurs manquantes comme certaines dataframe en possèdent encore.

Tout d'abord, on va joindre les dataframe meteo et airparif :

In [ ]:
df_meteo_jointure = df_meteo.reset_index(
    drop=True
)  # ligne nécessaire pour eviter un problème d'index dans la dataframe meteo

df_analyse = df_air_paris.merge(
    df_meteo_jointure,
    left_on="heure",
    right_on="time",
    how="left",
    validate="many_to_one",
)

df_analyse = df_analyse.drop(
    columns=["time"]
)  # etant donné que cette colone apparait alors 2 fois
display(df_analyse.head())

On étudie tout d'abord si, en générale, le vent pousse les particules de NO2 hors de paris les jours venteux. Pour ce faire, on se séparera des valeurs manquantes qui peuplent la colonne "no2", et on dessinera un nuage de point

In [ ]:
analyse_vent = df_analyse.dropna(subset=["no2", "wind_speed_10m"]).copy()

plt.figure(figsize=(10, 6))
plt.scatter(analyse_vent["wind_speed_10m"], analyse_vent["no2"], alpha=0.1, s=5)

plt.xlabel("Vitesse du vent à 10 m (km/h)")
plt.ylabel("Concentration de NO2 (µg/m³)")
plt.title("Concentration de NO2 selon la vitesse du vent")
plt.grid(alpha=0.2)
plt.show()


# on dessine ensuite une courbe. pour moyenner sur des tranches de vitesse de vent la quantité de NO2 dans l'air


# On créée tout d'abord des tranches de vitesse de vent : 0 à 5, 5 a 10...
maximum = analyse_vent["wind_speed_10m"].max()
bornes = range(0, int(maximum) + 10, 5)
analyse_vent["tranche_vent"] = pd.cut(analyse_vent["wind_speed_10m"], bins=bornes)

# Onfait la moyenne des concentration de NO2 par tranches
moyennes = analyse_vent.groupby("tranche_vent", observed=True).agg(
    vent_moyen=("wind_speed_10m", "mean"),
    no2_moyen=("no2", "mean"),
    nombre_mesures=("no2", "size"),
)

plt.figure(figsize=(10, 6))
plt.plot(moyennes["vent_moyen"], moyennes["no2_moyen"], marker="o")
plt.xlabel("Vitesse moyenne du vent (km/h)")
plt.ylabel("Concentration moyenne de NO₂ (µg/m³)")
plt.title("Concentration moyenne de NO₂ selon la vitesse du vent")
plt.grid(alpha=0.3)
plt.show()

On voit alors que le NO2 moyen varie avec la vitesse du vent dans paris. Pour valider notre hypothèse, on calcule leur corrélation empirique:

In [ ]:
print(
    "Corrélation de Pearson entre vitesse du vent et cincentration de NO2 dans l'athmosphère de Paris:",
    analyse_vent["wind_speed_10m"].corr(analyse_vent["no2"]),
)

Ici, on peut conclure une corrélationn négative moyenne entre les deux variables. 

On peut alors se demander si il n'y aurait pas d'autres variables qui pourraient influencer ce résultat. Par exemple, il pourrait être possible que le vent soufle plus fort la nuit que le jour. Or, la nuit, on aurait tendance à dire que la population polue moins. On peut donc essayer d'observer via le théorème du biais de la variable omise si la variable "is_day" à une influence sur la corrélation trouvée.


In [ ]:
import statsmodels.formula.api as smf

analyse_vent_jour_nuit = df_analyse.dropna(subset=["no2", "wind_speed_10m", "is_day"])

# Sans puis avec la variable jour/nuit
sans_jour = smf.ols("no2 ~ wind_speed_10m", data=analyse_vent_jour_nuit).fit()
avec_jour = smf.ols("no2 ~ wind_speed_10m + is_day", data=analyse_vent_jour_nuit).fit()

print("Coefficient du vent sans is_day :", sans_jour.params["wind_speed_10m"])
print("Coefficient du vent avec is_day :", avec_jour.params["wind_speed_10m"])

Ici, on a effectué deux régressions linéaire pour répondre à cette question. On voit que les coefficients de régressions calculés semblent proche, ce qui nous pousse à confirmer que le vent à un impact significatif sur la concentration de NO2 dans l'athmosphère indépendament du jour ou de la nuit. On peut alors effectuer cette même analyse sur les autres gazs du dataset air parif :

In [ ]:
analyse_vent_full = df_analyse.dropna(
    subset=["pm25", "pm10", "nox", "wind_speed_10m"]
).copy()

plt.figure(figsize=(10, 6))
plt.scatter(analyse_vent["wind_speed_10m"], analyse_vent["pm25"], alpha=0.1, s=5)

plt.xlabel("Vitesse du vent à 10 m (km/h)")
plt.ylabel("Concentration de pm2,5 (µg/m³)")
plt.title("Concentration de pm2,5 selon la vitesse du vent")
plt.grid(alpha=0.2)
plt.show()

plt.figure(figsize=(10, 6))
plt.scatter(analyse_vent["wind_speed_10m"], analyse_vent["pm10"], alpha=0.1, s=5)

plt.xlabel("Vitesse du vent à 10 m (km/h)")
plt.ylabel("Concentration de pm10 (µg/m³)")
plt.title("Concentration de pm10 selon la vitesse du vent")
plt.grid(alpha=0.2)
plt.show()

plt.figure(figsize=(10, 6))
plt.scatter(analyse_vent["wind_speed_10m"], analyse_vent["nox"], alpha=0.1, s=5)

plt.xlabel("Vitesse du vent à 10 m (km/h)")
plt.ylabel("Concentration de NOx (µg/m³)")
plt.title("Concentration de NOx selon la vitesse du vent")
plt.grid(alpha=0.2)
plt.show()


print(
    "Corrélation entre vitesse du vent et concentration de pm2,5 dans l'athmosphère de Paris:",
    analyse_vent["wind_speed_10m"].corr(analyse_vent["pm25"]),
)

print(
    "Corrélation entre vitesse du vent et concentration de pm10 dans l'athmosphère de Paris:",
    analyse_vent["wind_speed_10m"].corr(analyse_vent["pm10"]),
)

print(
    "Corrélation entre vitesse du vent et concentration de NOx dans l'athmosphère de Paris:",
    analyse_vent["wind_speed_10m"].corr(analyse_vent["nox"]),
)

On retrouve dans chacun des cas une corrélation négative, de faible à moyenne. 

On voit donc que la météo influence statistiquement la concentration de certaines particules dans l'athmosphère parisienne. Cependant, cela ne nous permet pas de conclure que les parisiens poluent moins lorsque le vent soufle sur la capitale. Nous allons voir cela à travers notre deuxième analyse. Cette fois ci, nous essayerons de comprendre à quelle point la pluie peut elle avoir une influence, certainement positive, sur la polution enregistrée à Paris.

In [ ]:
analyse_pluie = df_analyse.dropna(subset=["no2", "precipitation"]).copy()

# Comparaison entre heures sans pluie et avec pluie

# Nuage de points
plt.figure(figsize=(10, 6))
plt.scatter(analyse_pluie["precipitation"], analyse_pluie["no2"], alpha=0.05, s=5)
plt.xlabel("Précipitations horaires (mm)")
plt.ylabel("Concentration de NO2 (µg/m³)")
plt.title("NO₂ selon les précipitations")
plt.grid(alpha=0.2)
plt.show()

print(
    "Corrélation entre précipitation horaire et concentration de NO2 dans l'athmosphère de Paris:",
    analyse_pluie["wind_speed_10m"].corr(analyse_pluie["nox"]),
)

Cette corrélation négative ne va pas dans le sens de notre hypothèse. Pour expliquer ce phénomène, on peut supposer qu'il y a relativement plus de vent les jours pluvieux que les jours ensoleillés. Pour analyser cela, considérons la vitesse de vent moyenne les jours de vent pluvieux et les jours de vent non pluvieux :

In [ ]:
analyse_pluie["pleut_il_?"] = analyse_pluie["precipitation"] > 0

print(analyse_pluie.groupby("pleut_il_?")["wind_speed_10m"].mean())  # On regroupe en fonction de si il pleut ou pas

Il y a donc plus de vent lorsqu'il pleut, mais cette différence ne semble pas significative. On peut essayer de faire une nouuvelle régression linéaire, cette fois ci en régressant la quantité de NO2 dans l'athmosphère à Paris sur les précipitations ET la vitesse du vent : 

In [ ]:
modele_pluie_vent = smf.ols(
    "no2 ~ precipitation + wind_speed_10m", data=analyse_pluie
).fit()

print(
    "Coefficient des précipitations dans la régression : ",
    modele_pluie_vent.params["precipitation"],
)

On obtient, en tennant compte cette fois de la vitesse du vent que la pluie à en effet, dans un nouveau sens, une corrélation positive avec la polution au NO2 dans Paris. Pour finir cette partie, on se propose de; dans un premier temps trouver la directions dans lequel le vent soufle le plus en moyenne à Paris, d'ensuite comparer les valeurs de poulution au NOx enregistrés par deux capteurs opposés dans la direction du vent. Ainsi, on pourra voir si le vent déplace la polution en fonction des directions des vents, ou plutôt que le vent va étaler la polution sans réel lien avec la direction du vent. 

In [ ]:
meteo_vent = df_meteo.dropna(
    subset=["wind_direction_10m"]
).copy() 
direction = df_meteo["wind_direction_10m"].copy() % 360

#on va découper les directions du vent en secteur et les compter en fonction du vent

meteo_vent["direction_vent"] = pd.cut(
    (direction + 45) % 360,  #on place les axes 
    bins=[0, 90, 180, 270, 360],
    labels=["Nord", "Est", "Sud", "Ouest"],
    right=False,
)

frequence_direction = meteo_vent["direction_vent"].value_counts()
frequence_direction.plot.bar(
    title="Directions du vent les plus fréquentes à Paris",
    xlabel="Direction d'où vient le vent",
    ylabel="Nombre d'heures",
    rot=0,
)
plt.show()

Le vent provient en moyenne plus du sud. A partir de cela, on va extraire les données des deux stations suivantes : celle dans le 13eme arondissement et celle dans le 18eme arondissement (l'une au sud, l'autre au nord de paris)

In [ ]:
# Garder les mesures des stations du 18e et du 13e
comparaison_stations = df_analyse[
    df_analyse["station"].isin(["FR04004", "FR04037"])
].copy()

# Retirer les lignes sans mesure de NOx ou direction du vent
comparaison_stations = comparaison_stations.dropna(
    subset=["nox", "wind_direction_10m"]
)

nox_18e = comparaison_stations[
    comparaison_stations["station"] == "FR04004"
].set_index("heure")["nox"]

nox_13e = comparaison_stations[
    comparaison_stations["station"] == "FR04037"
].set_index("heure")["nox"]

ecart_nox = (nox_18e - nox_13e).dropna().to_frame("ecart_nox")

print(ecart_nox.mean())

On voit donc une légère augmentation du NOX dans le 18ème par rapport au 13ème. On peut donc dire que le vent pousse bien les particules poluantes dans sa direction.

On peut donc dire que la météo est liée à la pollution à Paris, notamment à travers le vent et la pluie. On peut alors se demander si les variations météorologiques influencent aussi la pollution parce que les Parisiens utilisent plus ou moins leur voiture lorsque les conditions météorologiques sont difficiles ?

### **C) Trafic et météo**

**Question.** À heure égale, les Parisiens roulent-ils moins quand il pleut, quand il gèle ou quand il fait très chaud ? Si oui, une partie du lien entre météo et pollution passerait par le trafic.

A noter qu'il y a une différence entre les deux datasets. Le trafic est mesuré rue par rue (1 531 arcs), la météo en un seul point pour tout Paris (Open-Meteo). On va donc ramèner le trafic à une série temporelle qui sera le **débit moyen des arcs, heure par heure**. On retire les heures où moins de la moitié des arcs sont mesurés (coupures de la collecte : la moyenne n'y porterait que sur quelques rues). Les deux séries se joignent ensuite sur `heure` (UTC).

On fera également attention au faite que le trafic dépend d'abord de l'heure et du jour : le lundi à 8 h il y a toujours plus de voitures que le dimanche à 4 h, quel que soit le temps. On compare donc chaque heure à ce qui est normal pour elle :

$$\text{débit relatif} = \frac{\text{débit de l'heure}}{\text{débit moyen de la même heure de la semaine (heure de Paris)}}$$

1 = trafic habituel ; 0,95 = 5 % de voitures en moins que d'habitude à ce moment de la semaine.

Le code est dans `src/analyse/trafic_meteo.py`, les seuils dans `config.yaml` (section `analyse.trafic_meteo`).

In [ ]:
from src.analyse import trafic_meteo

PARAM_TM = CFG["analyse"]["trafic_meteo"]

trafic_heure = trafic_meteo.trafic_par_heure(df_trafic_clean, PARAM_TM["part_arcs_min"])
trafic_heure = trafic_meteo.ajouter_debit_relatif(trafic_heure)
df_trafic_meteo = trafic_meteo.joindre_meteo(trafic_heure, df_meteo, PARAM_TM["seuil_pluie_mm"])

n_retirees = df_trafic_clean["heure"].nunique() - len(trafic_heure)
print(f"{len(df_trafic_meteo)} heures gardées, {n_retirees} retirées (moins de "
      f"{PARAM_TM['part_arcs_min']:.0%} des arcs mesurés : coupures de la collecte)")
print(f"Heures pluvieuses (> {PARAM_TM['seuil_pluie_mm']} mm) : {df_trafic_meteo['pluie'].sum()} "
      f"({df_trafic_meteo['pluie'].mean():.1%})")
display(df_trafic_meteo.head())

**Pluie.** On compare, heure de la journée par heure de la journée, le débit relatif des heures sèches et des heures pluvieuses (précipitations > 0,1 mm).

In [ ]:
effet_pluie = trafic_meteo.effet_pluie_par_heure(df_trafic_meteo)
moyenne_pluie = df_trafic_meteo.groupby("pluie")["debit_relatif"].mean()
print(f"Débit relatif moyen : {moyenne_pluie[False]:.3f} au sec, {moyenne_pluie[True]:.3f} sous la pluie "
      f"({moyenne_pluie[True] / moyenne_pluie[False] - 1:+.1%})")

fig, ax = plt.subplots(figsize=(12, 4.5), constrained_layout=True)
ax.axhline(1, color="grey", linewidth=0.8)
ax.plot(effet_pluie.index, effet_pluie["sec"], color="#d97706", marker="o", markersize=4, label="heures sèches")
ax.plot(effet_pluie.index, effet_pluie["pluie"], color="#2563eb", marker="o", markersize=4, label="heures pluvieuses")
ax.set_xticks(range(0, 24, 2))
ax.set_xlabel("Heure de la journée (heure de Paris)")
ax.set_ylabel("Débit relatif (1 = trafic habituel)")
ax.set_title("Sous la pluie, le trafic reste proche de son niveau habituel, un peu plus bas la nuit et en soirée")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.25)
plt.show()

**Température.** Même idée par tranche de 5 °C. Les heures très froides tombent surtout pendant les vacances de Noël, et les plus chaudes en été (vacances aussi) : la deuxième barre divise en plus par la moyenne du mois, pour retirer cet effet « calendrier ».

In [ ]:
effet_temperature = trafic_meteo.effet_par_tranche(
    df_trafic_meteo, "temperature_2m", PARAM_TM["bornes_temperature"]
)
display(effet_temperature.round(3))

etiquettes = [f"{int(t.left)} à {int(t.right)}\n({n} h)"
              for t, n in zip(effet_temperature.index, effet_temperature["heures"])]
x = np.arange(len(etiquettes))
fig, ax = plt.subplots(figsize=(12, 4.5), constrained_layout=True)
ax.axhline(1, color="grey", linewidth=0.8)
ax.bar(x - 0.2, effet_temperature["debit_relatif"], width=0.4, color="#2563eb", label="à heure de la semaine égale")
ax.bar(x + 0.2, effet_temperature["a_mois_egal"], width=0.4, color="#93c5fd", label="et à mois égal")
ax.set_xticks(x, etiquettes)
ax.set_ylim(0.85, 1.06)
ax.set_xlabel("Température (°C), avec le nombre d'heures de chaque tranche")
ax.set_ylabel("Débit relatif (1 = trafic habituel)")
ax.set_title("Le trafic baisse nettement quand il gèle, et un peu pendant les fortes chaleurs")
ax.legend(frameon=False, loc="upper right")
plt.show()

**Corrélations.** (elle ne suppose pas de relation linéaire et résiste aux valeurs extrêmes, comme les averses rares mais fortes).

In [ ]:
correlations_tm = (
    df_trafic_meteo[["debit_relatif", "precipitation", "temperature_2m", "wind_speed_10m"]]
    .corr(method="spearman")["debit_relatif"]
    .drop("debit_relatif")
    .rename({"precipitation": "précipitations (mm)", "temperature_2m": "température (°C)",
             "wind_speed_10m": "vent (km/h)"})
)
display(correlations_tm.round(3).to_frame("corrélation de avec le débit relatif"))

**Régression.** Les comparaisons précédentes regardent une variable à la fois : or quand il gèle, il peut aussi pleuvoir ou venter, et les jours froids tombent pendant les vacances de Noël. On estime donc tous les effets ensemble, par moindres carrés ordinaires :

$$\text{écart}_h = \beta_1\,\text{pluie}_h + \sum_k \gamma_k\,\mathbb{1}[\text{température}_h \in k] + \beta_2\,\text{vent}_h + \alpha_{\text{mois}(h)} + \varepsilon_h$$

- $\text{écart}_h = 100 \times (\text{débit relatif}_h - 1)$ : écart au trafic habituel de cette heure de la semaine, en % ;
- chaque coefficient se lit **toutes choses égales par ailleurs**, en points de % ; les tranches de température sont comparées à la tranche 10-15 °C ;
- $\alpha_{\text{mois}}$ (effets fixes du mois) absorbe les vacances et la saison ;
- deux heures qui se suivent se ressemblent (résidus autocorrélés) : les écarts-types ordinaires seraient trop petits, on utilise donc des écarts-types **HAC (Newey-West)** sur 24 heures.

In [ ]:
modele_tm = trafic_meteo.regression_meteo(
    df_trafic_meteo, PARAM_TM["bornes_temperature"], PARAM_TM["tranche_reference"], PARAM_TM["retards_hac"]
)
coefs_tm = trafic_meteo.tableau_regression(modele_tm)
print(f"{int(modele_tm.nobs)} heures, R² = {modele_tm.rsquared:.3f} (effets fixes du mois compris)")
display(coefs_tm.round(3))

y = np.arange(len(coefs_tm))[::-1]
significatif = coefs_tm["p-valeur"] < 0.05
fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
ax.axvline(0, color="grey", linewidth=0.8)
ax.errorbar(
    coefs_tm["effet (points de %)"], y,
    xerr=[coefs_tm["effet (points de %)"] - coefs_tm["IC 95 % bas"],
          coefs_tm["IC 95 % haut"] - coefs_tm["effet (points de %)"]],
    fmt="none", ecolor="#94a3b8", capsize=3,
)
ax.scatter(coefs_tm["effet (points de %)"], y, color=np.where(significatif, "#2563eb", "#94a3b8"), zorder=3)
ax.set_yticks(y, [nom + (" (réf. 10-15 °C)" if nom.startswith("température") else "") for nom in coefs_tm.index])
ax.set_xlabel("Effet sur le trafic, en % du trafic habituel (point bleu : significatif à 5 %)")
ax.set_title("Toutes choses égales par ailleurs, le gel et la canicule font baisser le trafic, la pluie très peu")
ax.grid(axis="x", alpha=0.25)
plt.show()

**Conclusions**

1. **La pluie ne réduit presque pas le trafic** : −1,4 % dans la régression (IC 95 % : −2,5 à −0,3), comme dans la comparaison simple (0,989 contre 1,001 au sec). L'effet est un peu plus marqué la nuit (jusqu'à −7 % vers minuit) et en soirée (−2 %), alors qu'il est légèrement plus haut en début d'après-midi (+2 %), mais ces moyennes reposent sur peu d'heures pluvieuses (15 à 50 par heure de la journée).
2. **Le froid fait nettement baisser le trafic** : −14 % sous 0 °C par rapport à 10-15 °C (IC 95 % large : −24 à −5, car seulement 220 heures), −5 % entre 0 et 5 °C. La régression contrôle le mois : ce n'est donc pas seulement l'effet des vacances de Noël.
3. **Les fortes chaleurs le font aussi baisser** : −5 % au-delà de 35 °C (IC 95 % : −7 à −3). Entre 15 et 30 °C, le trafic est au contraire un peu plus élevé (+2 à +4 %).
4. **Le vent n'a pas d'effet significatif** (−0,7 % pour 10 km/h, p = 0,23).
5. La météo explique peu les variations du trafic dans l'ensemble : les corrélations sont quasi nulles (|ρ| < 0,04) et le R² de la régression reste faible (0,20, effets fixes du mois compris). Les effets n'apparaissent qu'aux extrêmes (gel, canicule).

**Conséquences** Le lien météo vers pollution vu en A) ne passe donc que très peu par le trafic : le vent et la pluie agissent surtout directement, en dispersant et en lessivant les polluants.

### **D) Trafic des axes proches et NOx**

**Question.** Aux heures où il y a plus de voitures autour d'une station, mesure-t-elle plus d'oxydes d'azote (NOx) ? Le lien est-il plus fort aux stations de trafic, placées au bord des axes, qu'aux stations de fond ?

**Méthode.**
- Pour chaque station, on prend les arcs à moins de 500 m et on calcule leur **débit moyen heure par heure** (heure gardée si au moins la moitié de ces arcs est mesurée).
- On le joint au NOx de la station sur (`station`, `heure`) : 14 stations ont des arcs à moins de 500 m, 7 de trafic et 7 de fond.
- Le trafic et le NOx suivent tous deux le rythme de la journée. Comme en B), on calcule donc aussi leurs valeurs **relatives à la même heure de la semaine** : une corrélation « à heure égale » mesure si une heure plus chargée que d'habitude est aussi plus polluée que d'habitude.

In [ ]:
from src.analyse import nox_trafic

PARAM_NT = CFG["analyse"]["nox_trafic"]

liens_nt = nox_trafic.arcs_autour_des_stations(points_air, arcs_clean, PARAM_NT["rayon_m"])
trafic_stations = nox_trafic.trafic_autour_des_stations(df_trafic_clean, liens_nt, PARAM_NT["part_arcs_min"])
df_nox_trafic = nox_trafic.joindre_air(trafic_stations, df_air, points_air, PARAM_NT["polluant"])

print(f"{df_nox_trafic['station'].nunique()} stations, {len(df_nox_trafic):,} heures station".replace(",", " "))
display(
    df_nox_trafic.groupby(["influence", "nom", "station"]).size().rename("heures").reset_index()
    .merge(liens_nt.groupby("station").size().rename("arcs"), on="station")
)

**NOx selon le niveau de trafic.** Pour chaque station, on découpe les heures en 10 déciles de trafic (1 = les 10 % d'heures les plus calmes), puis on fait la moyenne du NOx par type de station.

In [ ]:
deciles_nt = nox_trafic.pollution_par_decile(df_nox_trafic)
display(deciles_nt.round(1))

fig, ax = plt.subplots(figsize=(10, 4.5), constrained_layout=True)
couleurs_nt = {"trafic": "#dc2626", "fond": "#2563eb"}
for influence in deciles_nt.columns:
    ax.plot(deciles_nt.index, deciles_nt[influence], marker="o", color=couleurs_nt[influence],
            label=f"stations de {influence}")
ax.set_xticks(deciles_nt.index)
ax.set_ylim(bottom=0)
ax.set_xlabel("Décile de trafic autour de la station (1 = heures les plus calmes)")
ax.set_ylabel("NOx moyen (µg/m³)")
ax.set_title("Au bord des axes, le NOx double quand le trafic augmente ; en fond, il varie peu")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.25)
plt.show()

**Corrélation station par station.** Corrélation entre trafic et NOx, brute puis à heure de la semaine égale.

In [ ]:
corr_nt = nox_trafic.correlations_par_station(df_nox_trafic)
display(corr_nt.round(2))
display(corr_nt.groupby("influence")[["brute", "a_heure_egale"]].median().round(2))

y = np.arange(len(corr_nt))
fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
ax.barh(y + 0.2, corr_nt["brute"], height=0.4, color=corr_nt["influence"].map(couleurs_nt), label="brute")
ax.barh(y - 0.2, corr_nt["a_heure_egale"], height=0.4, color=corr_nt["influence"].map(couleurs_nt),
        alpha=0.4, label="à heure de la semaine égale")
ax.axvline(0, color="grey", linewidth=0.8)
ax.set_yticks(y, corr_nt["nom"] + " (" + corr_nt["influence"] + ")")
ax.set_xlabel("Corrélation de trafic / NOx")
ax.set_title("Le lien est surtout porté par le rythme de la journée, et plus fort aux stations de trafic")
ax.legend(frameon=False, loc="lower right")
plt.show()

**Le choix du rayon change-t-il le résultat ?** Corrélations médianes par type de station pour 200, 500 et 1 000 m.

In [ ]:
display(nox_trafic.sensibilite_rayon(
    points_air, arcs_clean, df_trafic_clean, df_air, PARAM_NT["polluant"],
    PARAM_NT["rayons_sensibilite"], PARAM_NT["part_arcs_min"],
).round(2))

**Conclusions**

1. **Aux stations de trafic, le NOx suit nettement le trafic** : il passe de 30 µg/m³ aux heures les plus calmes à 71 µg/m³ aux plus chargées. Aux stations de fond, il passe seulement de 19 à 27 µg/m³ et ne bouge presque plus à partir du 4ᵉ décile. Corrélation médiane : 0,40 aux stations de trafic contre 0,16 en fond.
2. **À heure égale, le lien est beaucoup plus faible** (corrélation médiane de 0,11 en trafic, 0,10 en fond). L'essentiel du lien vient donc du rythme de la journée : heures de pointe chargées et polluées, nuits calmes. Une heure un peu plus chargée que d'habitude n'est que légèrement plus polluée.
3. **Place de l'Opéra et boulevard Haussmann ont une corrélation négative à heure égale.** Une hypothèse : sur ces axes souvent saturés, un embouteillage fait baisser le débit alors que les voitures, à l'arrêt, émettent plus. Le taux d'occupation (`k`) permettrait de le vérifier.
4. **Le résultat ne dépend pas du rayon** : les corrélations médianes sont presque identiques à 200, 500 et 1 000 m.

**Limites** Le débit moyen mélange les grands axes et petites rues. La météo (le vent surtout) fait varier le NOx indépendamment du trafic. Elle ajoute du bruit mais crée peu de faux lien, car on a vu en B) que le trafic dépend très peu de la météo.

**Conséquence pour la suite** Pour prédire le NOx, le trafic est surtout utile aux stations de trafic : le modèle devra tenir compte du type de station, et de l'heure de la journée qui porte une grande partie du lien.